# 165 — Privacidad, secretos y minimización de datos

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El contrato auditable (kind + evidence) ayuda con el punto **logs/observabilidad**:
registra evidencia estructurada de decisiones sin necesidad de volcar el contenido sensible crudo,
lo que reduce la exposición de PII en registros.


In [ ]:
result = run_lab("safety", seed=165)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
1x : 3/50  = 6.0 %
10x: 12/30 = 40.0 %
50x: 15/20 = 75.0 %
global: (3+12+15)/(50+30+20) = 30/100 = 30.0 %
```

La global (30 %) promedia grupos con riesgo muy distinto: oculta que los datos muy repetidos se
extraen el 75 % de las veces y que incluso los únicos (6 %) exponen a personas individuales. El
riesgo de privacidad se lee por frecuencia e identificabilidad, no por el promedio.

**Ejercicio 3.** El agente debería recibir solo `nombre` (o un identificador) y `numero_pedido` —
los campos necesarios para la finalidad "consultar estado". Evita: (1) exposición de DNI/dirección/
tarjeta en el prompt y en los logs, y (2) que una inyección o un error del modelo pueda regurgitar
esos datos, que nunca llegaron al contexto.

**Ejercicio 4.**

```text
(a) tokenización de secretos  (vault + referencia, el valor nunca entra al modelo)
(b) privacidad diferencial    (garantía formal sobre estadísticas agregadas)
(c) retención/TTL             (conservar 30 días y borrar)
(d) seudonimización           (reversible internamente con el mapa protegido)
```


In [ ]:
# Verificación numérica del Ejercicio 2
grupos = {"1x": (3, 50), "10x": (12, 30), "50x": (15, 20)}
tasas = {k: e / n for k, (e, n) in grupos.items()}
glob = sum(e for e, _ in grupos.values()) / sum(n for _, n in grupos.values())
for k, v in tasas.items():
    print(f"{k}: {v:.1%}")
print(f"global: {glob:.1%}")
assert tasas["50x"] == 0.75 and glob == 0.30


## Reflexión (guía)

1. Porque minimizar la pérdida sobre datos repetidos empuja al modelo a almacenarlos literalmente;
   con suficiente capacidad y repetición, memorizar es la forma óptima de reducir el error.
2. Los **logs de prompts con PII**: se cierra redactando/seudonimizando antes de escribir el log y
   aplicando retención. Se descuida porque los logs se ven como herramienta interna "de confianza".
3. Garantiza que la presencia o ausencia de un individuo apenas cambia la salida (acotado por
   epsilon); NO garantiza utilidad, ni protege contra fugas fuera del mecanismo con DP (p. ej. un
   log sin procesar).
